# 3、ConversationChain的使用



In [8]:
from langchain.chains.conversation.base import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
import os
import dotenv

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提供PromptTemplate结构
template = """
我是一个人工智能的助手，可以详细的回复用户的问题。

# 历史问题：{history}

当前问题：{input}

"""
prompt_template = PromptTemplate.from_template(template)

#3、提供ConversationChain的实例，此实例是对ConversationBufferMemory和LLMChain的封装
# memory = ConversationBufferMemory(memory_key="chat_history")
#
# llm_chain = LLMChain(
#     llm=chat_model,
#     prompt=prompt_template,
#     memory=memory,
# )

convers_chain = ConversationChain(
    llm = chat_model,
    prompt = prompt_template,
)

#5、调用过程
# 此时的key必须是input,否则报错
convers_chain.invoke({"input":"1 + 3 = ？"})


{'input': '1 + 3 = ？',
 'history': '',
 'response': '1 + 3 = 4。这个算式的结果是4。有什么其他问题或者需要进一步的帮助吗？'}

In [2]:
convers_chain.invoke({"input":"我刚才问了什么问题？"})

{'input': '我刚才问了什么问题？',
 'history': 'Human: 1 + 3 = ？\nAI: 1 + 3 = 4。',
 'response': '您刚才问的问题是“1 + 3 = ？”我回答说“1 + 3 = 4”。如果您还有其他问题或者需要进一步的帮助，请随时告诉我！'}

使用默认的提示词，演示ConversationChain的使用

In [3]:
#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提示词模板：略。因为使用ConversationChain的默认提示词模板

#3、创建ConversationChain的实例
chain = ConversationChain(
    llm = chat_model,
)

#4、调用
resp1 = chain.invoke({"input":"3 + 2 = ?"})
print(resp1)

resp2 = chain.invoke({"input":"我刚才问了什么问题？"})
print(resp2)

{'input': '3 + 2 = ?', 'history': '', 'response': "3 + 2 = 5! It's a simple arithmetic problem. If you think about it, when you have three apples and you add two more apples, you end up with a total of five apples. Do you want to know more about basic math or perhaps something else?"}
{'input': '我刚才问了什么问题？', 'history': "Human: 3 + 2 = ?\nAI: 3 + 2 = 5! It's a simple arithmetic problem. If you think about it, when you have three apples and you add two more apples, you end up with a total of five apples. Do you want to know more about basic math or perhaps something else?", 'response': '你刚才问的问题是 "3 + 2 = ?" 你想了解更多关于数学的内容，还是有其他问题想问我呢？'}


# 4、ConversationBufferWindowMemory的使用

特点：保留最近的k条记录作为memory提供给大模型使用

① 基本使用

In [4]:
# 1.导入相关包
from langchain.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k = 1)
# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手"})
memory.save_context({"input": "你的生日是哪天？"}, {"output": "我不清楚"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': 'Human: 你的生日是哪天？\nAI: 我不清楚'}


C:\Users\41507\AppData\Local\Temp\ipykernel_24100\2247884119.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k = 1)


使用PromptTemplate，演示ConversationBufferWindowMemory的使用

In [5]:
from langchain.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate

#1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)

#2、提供PromptTemplate结构
template = """
我是一个人工智能的助手，可以详细的回复用户的问题。

历史问题：{history}

当前问题：{question}

"""
prompt_template = PromptTemplate.from_template(template)

#3、提供ConversationBufferWindowMemory的实例
# memory = ConversationBufferWindowMemory(k=1)
memory = ConversationBufferWindowMemory(k=3)

#4、提供LLMChain的实例
llm_chain = LLMChain(
    llm=chat_model,
    prompt=prompt_template,
    memory=memory,
)

#5、调用过程
res1 = llm_chain.invoke({"question": "你好，我的名字叫小明"})
res2 = llm_chain.invoke({"question": "今天的天气不错，适合做啥呢？"})
res2 = llm_chain.invoke({"question": "今年高考我考了681，我不去北大，我选择福耀科技大学"})
res4 = llm_chain.invoke({"question": "我的名字叫啥？"})
print(res4)

C:\Users\41507\AppData\Local\Temp\ipykernel_24100\535363192.py:25: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


{'question': '我的名字叫啥？', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你。有任何问题或者需要帮助的地方，请随时告诉我！\nHuman: 今天的天气不错，适合做啥呢？\nAI: 你好，小明！今天的天气不错，确实很适合进行一些户外活动。你可以考虑以下几种选择：\n\n1. **散步或跑步**：在公园或者附近的步道上散步或者慢跑，享受大自然的美好。\n\n2. **骑自行车**：如果你有自行车，可以骑行在城市的自行车道上，享受风景。\n\n3. **野餐**：带上一些食物和饮料，约上朋友或者家人，去公园进行一次野餐。\n\n4. **登山或徒步旅行**：如果附近有山或者自然保护区，可以去徒步旅行，锻炼身体的同时欣赏美景。\n\n5. **拍照**：如果你喜欢摄影，可以带上相机，去拍一些风景照或者街拍，记录下美好的一天。\n\n6. **参加户外活动**：可以看看有没有附近的户外活动，比如集市、音乐会或者运动赛事等。\n\n希望你能享受这个美好的天气！如果你有其他的兴趣或者想法，也可以告诉我。\nHuman: 今年高考我考了681，我不去北大，我选择福耀科技大学\nAI: 恭喜你，高考取得了681的好成绩！这个分数非常不错，能选择的学校和专业也很多。福耀科技大学是一所值得关注的高校，特别是在工程和科技领域有着不错的声誉。选择适合自己的学校和专业才是最重要的，祝你在福耀科技大学的学习生活中一切顺利，收获满满！\n\n如果你对福耀科技大学的专业选择、校园生活或者其他方面有任何问题，欢迎随时问我！', 'text': '你刚刚告诉我你的名字是小明！如果你有其他问题或者需要帮助的地方，请随时告诉我！'}


小结：通过ConversationBufferWindowMemory(k=1) 或ConversationBufferWindowMemory(k=3) 体会读取不同条目数的message给到提示词模板